# 15. QED 롤백 안전장치 추가

## 이번 노트북에서 할 것
- 세션 1(최초 설계 논의)에서 계획했으나 미뤄뒀던 "QED 롤백" 구현
- 치환 시 독성은 개선되나 약물유사성(QED)이 크게 떨어지는 경우 감지 후 롤백
- iterative_fix_loop에 통합, 회귀 테스트로 기존 동작 유지 확인
- (돌려두고 제안서 작업 병행) 여유 있으면 전체 held-out(1174개) 규칙기반 대량 실행

## 간략한 정리 (14까지)
- 라이브러리 11개 규칙, 전 규칙 후보 2개 이상 확보 완료
- 3-endpoint(Tox21/Ames/hERG) 검증 체계 완성
- 규칙기반 vs LLM기반 비교: 30%(7/23)에서 다른 후보 선택, Ames에서 LLM 우위
  가장 뚜렷(5/7), Tox21/hERG는 혼재
- 설계 단계에서 계획했던 QED 롤백이 아직 미구현 상태로 남아있었음을 확인

## 다음에 해야 할 것 (오늘 끝나면)
- quinone_A 등 고리형 어려운 규칙 도전 (16번)
- 전체 held-out set 최종 대량 실행 (17번)
- Case B를 3+ attachment point로 일반화 (여유 시)

In [1]:
# 셀 1
!pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 37.2 MB/s eta 0:00:00


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 168 (delta 77), reused 119 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 336.76 KiB | 3.70 MiB/s, done.
Resolving deltas: 100% (77/77), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
import importlib
from rdkit import Chem
from rdkit.Chem import QED
from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop
from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

data = load_tox21_clean()
print("도구 로드 확인 완료")

# QED 계산 테스트
test_mol = Chem.MolFromSmiles("CCO")
print("QED 테스트:", QED.qed(test_mol))

[02:30:19] WARNING: not removing hydrogen atom without neighbors
[02:30:19] Explicit valence for atom # 8 Al, 6, is greater than permitted
[02:30:19] Explicit valence for atom # 3 Al, 6, is greater than permitted
[02:30:19] Explicit valence for atom # 4 Al, 6, is greater than permitted
[02:30:20] Explicit valence for atom # 4 Al, 6, is greater than permitted
[02:30:20] Explicit valence for atom # 9 Al, 6, is greater than permitted
[02:30:20] Explicit valence for atom # 5 Al, 6, is greater than permitted
[02:30:20] Explicit valence for atom # 16 Al, 6, is greater than permitted
[02:30:20] Explicit valence for atom # 20 Al, 6, is greater than permitted
[02:30:20] WARNING: not removing hydrogen atom without neighbors


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개
도구 로드 확인 완료
QED 테스트: 0.4068079656553945


In [5]:
def check_qed_acceptable(original_smiles, new_smiles, max_qed_drop=0.1):
    """치환 전후 QED를 비교해, 하락폭이 임계치 이내인지 확인."""
    orig_mol = Chem.MolFromSmiles(original_smiles)
    new_mol = Chem.MolFromSmiles(new_smiles)
    if orig_mol is None or new_mol is None:
        return True, None, None  # 계산 불가 시 안전하게 통과 처리 (다른 검증에 맡김)

    orig_qed = QED.qed(orig_mol)
    new_qed = QED.qed(new_mol)
    drop = orig_qed - new_qed

    acceptable = drop <= max_qed_drop
    return acceptable, orig_qed, new_qed

In [6]:
def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini",
                        max_qed_drop: float = 0.1):
    """진단->치환->재평가를 반복. QED가 임계치 이상 떨어지면 그 치환을 기각하고
    다른 후보를 시도(같은 규칙 내 후보가 더 있으면), 없으면 stuck으로 종료."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []
    qed_rejections = []  # QED 때문에 기각된 시도 기록

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "qed_rejections": qed_rejections}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
        unknown_problems = [p for p in problems if p not in known_problems]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "qed_rejections": qed_rejections}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems, client_type=llm_client_type)
            target_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')
        else:
            target_rule = known_problems[0]['rule_name']
            problem_reason = "규칙 기반(리스트 순서대로)"

        info = get_replacement_candidates(target_rule)
        n_candidates = len(info['candidates'])

        # QED 통과하는 후보를 찾을 때까지 순서대로 시도
        fixed = None
        chosen_idx = None
        candidate_reason = None
        for try_idx in range(n_candidates):
            if llm_client is not None and try_idx == 0:
                candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, target_rule, client_type=llm_client_type)
                idx_to_try = candidate_decision['candidate_idx']
                reason_to_use = candidate_decision.get('reason', '')
            else:
                idx_to_try = try_idx if llm_client is None else (try_idx + 1) % n_candidates
                reason_to_use = "규칙 기반(고정 인덱스)" if llm_client is None else "QED 기각 후 대체 후보"

            candidate_fixed = propose_fix(current, target_rule, idx_to_try)
            if candidate_fixed is None or not candidate_fixed['is_valid']:
                continue

            qed_ok, orig_qed, new_qed = check_qed_acceptable(current, candidate_fixed['new_smiles'], max_qed_drop)
            if qed_ok:
                fixed = candidate_fixed
                chosen_idx = idx_to_try
                candidate_reason = reason_to_use
                break
            else:
                qed_rejections.append({
                    "step": step, "rule": target_rule, "candidate_idx": idx_to_try,
                    "orig_qed": orig_qed, "new_qed": new_qed
                })
            if llm_client is None:
                break  # 규칙기반은 재시도 없이 첫 시도만 (기존 동작 유지)

        if fixed is None:
            return {"status": "stuck", "reason": f"'{target_rule}' 치환 실패 또는 QED 기준 미달",
                    "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "qed_rejections": qed_rejections}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "qed_rejections": qed_rejections}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step, "smiles": current, "fixed_rule": target_rule,
            "problem_reason": problem_reason, "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history,
            "skipped_rules": skipped_rules, "qed_rejections": qed_rejections}

In [8]:
sample_smiles_for_qed = []
for s in data['smiles_test'][:200]:
    p = detect_toxicophores(s)
    known = [x for x in p if get_replacement_candidates(x['rule_name']) is not None]
    if known:
        sample_smiles_for_qed.append((s, known[0]['rule_name']))
    if len(sample_smiles_for_qed) >= 10:
        break

for smi, rule in sample_smiles_for_qed:
    mol = Chem.MolFromSmiles(smi)
    print(f"[{rule}] QED={QED.qed(mol):.3f}  {smi[:50]}")

[imine_1] QED=0.577  O=C=NC1CCC(CC2CCC(N=C=O)CC2)CC1
[alkyl_halide] QED=0.399  C[C@]12CC[C@@H]3c4ccc(OC(=O)N(CCCl)CCCl)cc4CC[C@H]
[aniline] QED=0.481  Nc1ccc(C(=O)OCCCOC(=O)c2ccc(N)cc2)cc1
[aniline] QED=0.549  CC(C)(C)OC(=O)c1cccc(N)c1
[aldehyde] QED=0.595  CN(C)c1ccc(C=O)cc1
[Sulfonic_acid_2] QED=0.479  CC(C)COC(=O)CC(C(=O)OCC(C)C)S(=O)(=O)[O-]
[aldehyde] QED=0.307  O=C[O-]
[aniline] QED=0.683  CC(=O)NS(=O)(=O)c1ccc(N)cc1
[Sulfonic_acid_2] QED=0.456  NCCS(=O)(=O)O
[aniline] QED=0.773  Cc1cc(Cc2ccc(N)c(C)c2)ccc1N


In [10]:
for smi, rule in sample_smiles_for_qed:
    fixed = propose_fix(smi, rule, candidate_idx=0)
    if fixed is None or not fixed['is_valid']:
        print(f"[{rule}] 치환 실패: {smi[:40]}")
        continue

    orig_mol = Chem.MolFromSmiles(smi)
    new_mol = Chem.MolFromSmiles(fixed['new_smiles'])
    orig_qed = QED.qed(orig_mol)
    new_qed = QED.qed(new_mol)

    print(f"[{rule}] QED {orig_qed:.3f} -> {new_qed:.3f} (변화 {new_qed-orig_qed:+.3f})  candidate={fixed['candidate_used']}")

[imine_1] 치환 실패: O=C=NC1CCC(CC2CCC(N=C=O)CC2)CC1
[alkyl_halide] QED 0.399 -> 0.452 (변화 +0.053)  candidate=hydroxyl (alcohol)
[aniline] QED 0.481 -> 0.449 (변화 -0.032)  candidate=acetamide (acylated amine)
[aniline] QED 0.549 -> 0.773 (변화 +0.224)  candidate=acetamide (acylated amine)
[aldehyde] QED 0.595 -> 0.704 (변화 +0.109)  candidate=amide
[Sulfonic_acid_2] QED 0.479 -> 0.651 (변화 +0.172)  candidate=sulfonamide
[aldehyde] 치환 실패: O=C[O-]
[aniline] QED 0.683 -> 0.752 (변화 +0.069)  candidate=acetamide (acylated amine)
[Sulfonic_acid_2] QED 0.456 -> 0.457 (변화 +0.001)  candidate=sulfonamide
[aniline] QED 0.773 -> 0.826 (변화 +0.053)  candidate=acetamide (acylated amine)


In [11]:
multi_test_qed = None
for s in data['smiles_test']:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count >= 2:
        multi_test_qed = s
        break

print("테스트 분자:", multi_test_qed)

result_qed_check = iterative_fix_loop(multi_test_qed, max_iterations=10)  # 규칙기반으로 (API 호출 없이 빠르게)
print("상태:", result_qed_check['status'])

orig_qed_val = QED.qed(Chem.MolFromSmiles(multi_test_qed))
print(f"\n원본 QED: {orig_qed_val:.3f}")
for h in result_qed_check['history']:
    mol = Chem.MolFromSmiles(h['smiles'])
    if mol is None:
        continue
    q = QED.qed(mol)
    print(f"step {h['step']}: QED={q:.3f}  ({h['smiles'][:50]})")

테스트 분자: Nc1c(S(=O)(=O)[O-])cc(Br)c2c1C(=O)c1ccccc1C2=O
상태: no_known_fix

원본 QED: 0.507
step 0: QED=0.507  (Nc1c(S(=O)(=O)[O-])cc(Br)c2c1C(=O)c1ccccc1C2=O)
step 1: QED=0.633  (NC(=O)c1c(S(=O)(=O)[O-])cc(Br)c2c1C(=O)c1ccccc1C2=)
step 2: QED=0.650  (NC(=O)c1c(S(N)(=O)=O)cc(Br)c2c1C(=O)c1ccccc1C2=O)


In [13]:
import numpy as np

multi_test_pool = []
for s in data['smiles_test']:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count >= 2:
        multi_test_pool.append(s)
    if len(multi_test_pool) >= 15:
        break

print(f"다중 문제 분자 풀: {len(multi_test_pool)}개\n")

qed_drops = []
for s in multi_test_pool:
    result = iterative_fix_loop(s, max_iterations=10)
    orig_mol = Chem.MolFromSmiles(s)
    final_mol = Chem.MolFromSmiles(result['final_smiles'])
    if orig_mol is None or final_mol is None:
        continue
    orig_q = QED.qed(orig_mol)
    final_q = QED.qed(final_mol)
    drop = orig_q - final_q
    qed_drops.append(drop)
    n_steps = len(result['history']) - 1
    print(f"[{result['status']}, {n_steps}스텝] QED {orig_q:.3f} -> {final_q:.3f} (변화 {final_q-orig_q:+.3f})")

print(f"\n최대 QED 하락폭: {max(qed_drops):.3f}")
print(f"평균 QED 변화: {np.mean(qed_drops):+.3f}")
print(f"0.1 이상 하락한 경우: {sum(1 for d in qed_drops if d > 0.1)}개")

다중 문제 분자 풀: 15개

[no_known_fix, 2스텝] QED 0.507 -> 0.650 (변화 +0.143)
[stuck, 0스텝] QED 0.355 -> 0.355 (변화 +0.000)
[success, 3스텝] QED 0.360 -> 0.638 (변화 +0.279)
[success, 3스텝] QED 0.364 -> 0.638 (변화 +0.274)
[success, 2스텝] QED 0.504 -> 0.708 (변화 +0.205)
[stuck, 0스텝] QED 0.371 -> 0.371 (변화 +0.000)
[success, 2스텝] QED 0.559 -> 0.754 (변화 +0.195)
[stuck, 0스텝] QED 0.437 -> 0.437 (변화 +0.000)
[success, 3스텝] QED 0.433 -> 0.637 (변화 +0.205)
[success, 3스텝] QED 0.376 -> 0.677 (변화 +0.301)
[success, 3스텝] QED 0.408 -> 0.702 (변화 +0.294)
[success, 3스텝] QED 0.389 -> 0.637 (변화 +0.248)
[stuck, 0스텝] QED 0.290 -> 0.290 (변화 +0.000)
[success, 3스텝] QED 0.376 -> 0.677 (변화 +0.301)
[success, 3스텝] QED 0.395 -> 0.668 (변화 +0.273)

최대 QED 하락폭: 0.000
평균 QED 변화: -0.181
0.1 이상 하락한 경우: 0개


In [14]:
step2_changes = []
step3_changes = []

for s in multi_test_pool:
    result = iterative_fix_loop(s, max_iterations=10)
    orig_mol = Chem.MolFromSmiles(s)
    final_mol = Chem.MolFromSmiles(result['final_smiles'])
    if orig_mol is None or final_mol is None:
        continue
    n_steps = len(result['history']) - 1
    change = QED.qed(final_mol) - QED.qed(orig_mol)
    if n_steps == 2:
        step2_changes.append(change)
    elif n_steps == 3:
        step3_changes.append(change)

print(f"2스텝 평균 QED 변화: {np.mean(step2_changes):+.3f} (n={len(step2_changes)})")
print(f"3스텝 평균 QED 변화: {np.mean(step3_changes):+.3f} (n={len(step3_changes)})")
print(f"\n2스텝당 평균 개선폭: {np.mean(step2_changes)/2:+.4f}")
print(f"3스텝당 평균 개선폭: {np.mean(step3_changes)/3:+.4f}")

2스텝 평균 QED 변화: +0.181 (n=3)
3스텝 평균 QED 변화: +0.272 (n=8)

2스텝당 평균 개선폭: +0.0905
3스텝당 평균 개선폭: +0.0906
